# 04 — Exploración geoespacial: cuencas sedimentarias productivas × SESCO

Validación controlada de la capa geoespacial de cuencas productivas frente al dataset unificado SESCO.

**Fuentes:**
- `exploration/data/raw/cuencas_sedimentarias_productivas.csv`
- `exploration/data/processed/sesco_produccion_model_clean.csv`
- (referencia) `exploration/data/processed/sesco_latest_periods_by_view.csv`

**Alcance:** solo `agrupador_tipo = "cuenca"`. Petróleo y gas se analizan por separado (unidades distintas).

**No modifica** `streamlit_app/` — exploración primero, integración después.


## 1. Importar librerías

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import geopandas as gpd
import pandas as pd
import plotly.express as px
from shapely import wkt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## 2. Rutas y archivos fuente

In [ ]:
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "exploration").exists()), Path.cwd())

EXPLORATION_DIR = ROOT / "exploration"
RAW_DIR = EXPLORATION_DIR / "data" / "raw"
PROCESSED_DIR = EXPLORATION_DIR / "data" / "processed"

CUENCAS_RAW_PATH = RAW_DIR / "cuencas_sedimentarias_productivas.csv"
# Respaldo si el CSV aún no fue copiado a exploration/data/raw
CUENCAS_RAW_FALLBACK = ROOT / "data" / "raw" / "cuencas_sedimentarias_productivas.csv"

SESCO_PATH = PROCESSED_DIR / "sesco_produccion_model_clean.csv"
LATEST_PERIODS_PATH = PROCESSED_DIR / "sesco_latest_periods_by_view.csv"

GEOJSON_OUT = PROCESSED_DIR / "cuencas_productivas_geo.geojson"
JOIN_PREVIEW_OUT = PROCESSED_DIR / "cuencas_productivas_sesco_join_preview.csv"

CRS_DEFAULT = "EPSG:4326"
REQUIRED_GEO_COLS = ["WKT", "cuenca", "ubicacion", "tipo"]

print(f"ROOT: {ROOT}")
print(f"Cuencas (esperado): {CUENCAS_RAW_PATH}")
print(f"SESCO: {SESCO_PATH}")


## 3. Leer CSV de cuencas

In [ ]:
if CUENCAS_RAW_PATH.exists():
    cuencas_source = CUENCAS_RAW_PATH
elif CUENCAS_RAW_FALLBACK.exists():
    cuencas_source = CUENCAS_RAW_FALLBACK
    print(f"AVISO: usando respaldo en {CUENCAS_RAW_FALLBACK}")
else:
    raise FileNotFoundError(
        f"No se encontró cuencas_sedimentarias_productivas.csv en {CUENCAS_RAW_PATH} "
        f"ni en {CUENCAS_RAW_FALLBACK}"
    )

df_cuencas_raw = pd.read_csv(cuencas_source)
print(f"Fuente cuencas: {cuencas_source}")
print(f"Registros: {len(df_cuencas_raw):,}")
df_cuencas_raw.head()


## 4. Validar estructura del CSV geoespacial

In [ ]:
missing_cols = [c for c in REQUIRED_GEO_COLS if c not in df_cuencas_raw.columns]
if missing_cols:
    raise ValueError(f"Columnas faltantes en cuencas: {missing_cols}")

print("Columnas OK:", REQUIRED_GEO_COLS)
print(f"\nCantidad de registros: {len(df_cuencas_raw):,}")
print("\nValores únicos — cuenca:")
print(sorted(df_cuencas_raw["cuenca"].dropna().unique()))
print("\nValores únicos — tipo:")
print(sorted(df_cuencas_raw["tipo"].dropna().unique()))
print("\nValores únicos — ubicacion:")
print(sorted(df_cuencas_raw["ubicacion"].dropna().unique()))

print("\nNulos por columna:")
display(df_cuencas_raw[REQUIRED_GEO_COLS].isna().sum().to_frame("nulos"))

dup_cuenca = df_cuencas_raw["cuenca"].duplicated().sum()
print(f"\nDuplicados por cuenca: {dup_cuenca}")
if dup_cuenca:
    display(df_cuencas_raw[df_cuencas_raw["cuenca"].duplicated(keep=False)].sort_values("cuenca"))


## 5. WKT → geometría y GeoDataFrame

In [ ]:
def parse_wkt(series: pd.Series):
    return series.apply(lambda x: wkt.loads(x) if pd.notna(x) and str(x).strip() else None)

geometries = parse_wkt(df_cuencas_raw["WKT"])
if geometries.isna().any():
    raise ValueError("Hay filas sin geometría parseable desde WKT.")

gdf_cuencas = gpd.GeoDataFrame(
    df_cuencas_raw.drop(columns=["WKT"]).copy(),
    geometry=geometries,
    crs=CRS_DEFAULT,
)

print(f"CRS asignado: {gdf_cuencas.crs}")
print("\nTipos de geometría:")
print(gdf_cuencas.geometry.geom_type.value_counts())

invalid = (~gdf_cuencas.geometry.is_valid).sum()
print(f"\nGeometrías inválidas (is_valid=False): {invalid}")
if invalid:
    display(gdf_cuencas.loc[~gdf_cuencas.geometry.is_valid, ["cuenca", "ubicacion", "tipo"]])

bounds = gdf_cuencas.total_bounds  # minx, miny, maxx, maxy
print(f"\nBounds generales [minx, miny, maxx, maxy]: {bounds}")
gdf_cuencas.head()


## 6. Leer dataset SESCO unificado

In [ ]:
if not SESCO_PATH.exists():
    raise FileNotFoundError(f"No existe {SESCO_PATH}")

df_sesco = pd.read_csv(SESCO_PATH)
df_sesco["periodo_dt"] = pd.to_datetime(df_sesco["periodo_dt"], errors="coerce")

print(f"Filas SESCO: {len(df_sesco):,}")
print("Columnas:", list(df_sesco.columns))
df_sesco.head(3)


## 7. Filtrar SESCO por cuenca (petróleo y gas por separado)

In [ ]:
df_cuenca = df_sesco[df_sesco["agrupador_tipo"] == "cuenca"].copy()
print(f"Filas con agrupador_tipo='cuenca': {len(df_cuenca):,}")

cuencas_sesco = sorted(df_cuenca["agrupador_nombre"].dropna().unique())
print(f"\nCuencas en SESCO ({len(cuencas_sesco)}):")
print(cuencas_sesco)

for producto in ["petroleo", "gas"]:
    sub = df_cuenca[df_cuenca["producto"] == producto]
    periodos = sorted(sub["periodo_str"].dropna().unique())
    print(f"\n--- {producto} ---")
    print(f"Períodos disponibles: {len(periodos)} (min={periodos[0]}, max={periodos[-1]})")


## 8. Último período válido por producto + agrupador_tipo

In [ ]:
if LATEST_PERIODS_PATH.exists():
    df_latest = pd.read_csv(LATEST_PERIODS_PATH)
    latest_cuenca = (
        df_latest[df_latest["agrupador_tipo"] == "cuenca"]
        .set_index("producto")["latest_valid_period"]
        .to_dict()
    )
    print(f"Últimos períodos (archivo {LATEST_PERIODS_PATH.name}):")
    print(latest_cuenca)
else:
    latest_cuenca = {}
    for producto in ["petroleo", "gas"]:
        sub = df_cuenca[df_cuenca["producto"] == producto]
        latest_cuenca[producto] = sub["periodo_str"].max()
    print("Últimos períodos calculados desde el dataset:")
    print(latest_cuenca)

for producto, periodo in latest_cuenca.items():
    n = len(
        df_cuenca[
            (df_cuenca["producto"] == producto) & (df_cuenca["periodo_str"] == periodo)
        ]["agrupador_nombre"].unique()
    )
    print(f"  {producto} @ {periodo}: {n} cuencas con registro")


## 9. Normalización mínima de nombres de cuenca

In [ ]:
def normalize_cuenca_name(value: str) -> str:
    """Mayúsculas, sin tildes, espacios simples, sin prefijo CUENCA."""
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.upper().strip()
    text = re.sub(r"\s+", " ", text)
    if text.startswith("CUENCA "):
        text = text[7:].strip()
    return text

gdf_cuencas["cuenca_norm"] = gdf_cuencas["cuenca"].map(normalize_cuenca_name)
df_cuenca["agrupador_nombre_norm"] = df_cuenca["agrupador_nombre"].map(normalize_cuenca_name)

print("Ejemplo geo:", gdf_cuencas[["cuenca", "cuenca_norm"]].drop_duplicates())
print("\nEjemplo SESCO (muestra):")
display(
    df_cuenca[["agrupador_nombre", "agrupador_nombre_norm"]]
    .drop_duplicates()
    .head(10)
)


## 10. Comparar nombres de cuenca

In [ ]:
geo_names = sorted(gdf_cuencas["cuenca"].unique())
sesco_names = cuencas_sesco

geo_norm_set = set(gdf_cuencas["cuenca_norm"])
sesco_norm_set = set(df_cuenca["agrupador_nombre_norm"])

exactas_orig = sorted(set(geo_names) & set(sesco_names))
exactas_norm = sorted(geo_norm_set & sesco_norm_set)

solo_geo_orig = sorted(set(geo_names) - set(sesco_names))
solo_sesco_orig = sorted(set(sesco_names) - set(geo_names))
solo_geo_norm = sorted(geo_norm_set - sesco_norm_set)
solo_sesco_norm = sorted(sesco_norm_set - geo_norm_set)

print("=== Nombres originales ===")
print(f"Coincidencias exactas ({len(exactas_orig)}):", exactas_orig)
print(f"Solo en capa geo ({len(solo_geo_orig)}):", solo_geo_orig)
print(f"Solo en SESCO ({len(solo_sesco_orig)}):", solo_sesco_orig)

print("\n=== Tras normalización (sin equivalencias manuales) ===")
print(f"Coincidencias ({len(exactas_norm)}):", exactas_norm)
print(f"Solo geo norm ({len(solo_geo_norm)}):", solo_geo_norm)
print(f"Solo SESCO norm ({len(solo_sesco_norm)}):", solo_sesco_norm)

# Tabla de equivalencias explícitas (geo nombre original -> SESCO agrupador_nombre)
# NO forzar merge silencioso: documentar cada mapeo propuesto.
CUENCA_EQUIVALENCIAS_GEO_A_SESCO = {
    # La capa llama AUSTRAL MARINA; SESCO reporta AUSTRAL para producción por cuenca.
    "AUSTRAL MARINA": "AUSTRAL",
}

equiv_rows = []
for geo_name, sesco_name in CUENCA_EQUIVALENCIAS_GEO_A_SESCO.items():
    equiv_rows.append({
        "cuenca_geo": geo_name,
        "cuenca_sesco_propuesta": sesco_name,
        "geo_norm": normalize_cuenca_name(geo_name),
        "sesco_norm": normalize_cuenca_name(sesco_name),
        "sesco_existe": sesco_name in sesco_names,
    })

df_equivalencias = pd.DataFrame(equiv_rows)
print("\n=== Equivalencias propuestas (geo → SESCO) ===")
display(df_equivalencias)

gdf_cuencas["cuenca_join_key"] = gdf_cuencas["cuenca"].map(
    lambda x: normalize_cuenca_name(CUENCA_EQUIVALENCIAS_GEO_A_SESCO.get(x, x))
)


## 11. Merge exploratorio

In [ ]:
def produccion_ultimo_periodo(producto: str) -> pd.DataFrame:
    periodo = latest_cuenca[producto]
    sub = df_cuenca[
        (df_cuenca["producto"] == producto) & (df_cuenca["periodo_str"] == periodo)
    ][["agrupador_nombre", "agrupador_nombre_norm", "produccion", "producto", "periodo_str"]].copy()
    # Una fila por cuenca en el último período
    sub = sub.drop_duplicates(subset=["agrupador_nombre_norm"], keep="last")
    return sub, periodo

def merge_geo_sesco(producto: str) -> gpd.GeoDataFrame:
    prod_df, periodo = produccion_ultimo_periodo(producto)
    merged = gdf_cuencas.merge(
        prod_df,
        left_on="cuenca_join_key",
        right_on="agrupador_nombre_norm",
        how="left",
        indicator=True,
        suffixes=("", "_sesco"),
    )
    merged["producto"] = producto
    merged["periodo_str"] = periodo
    return merged

gdf_petroleo = merge_geo_sesco("petroleo")
gdf_gas = merge_geo_sesco("gas")

print("Último período petróleo:", latest_cuenca["petroleo"])
print("Último período gas:", latest_cuenca["gas"])


## 12. Validar resultado del merge

In [ ]:
def resumen_merge(gdf_merged: gpd.GeoDataFrame, producto: str):
    con_prod = gdf_merged["produccion"].notna().sum()
    sin_prod = gdf_merged["produccion"].isna().sum()
    print(f"\n=== {producto} ===")
    print(f"Cuencas en capa geo: {len(gdf_merged)}")
    print(f"Con producción asociada: {con_prod}")
    print(f"Sin producción (geo sin match SESCO): {sin_prod}")

    prod_df, periodo = produccion_ultimo_periodo(producto)
    matched_keys = set(gdf_merged.loc[gdf_merged["produccion"].notna(), "cuenca_join_key"])
    sin_geom = prod_df[~prod_df["agrupador_nombre_norm"].isin(matched_keys)]
    print(f"Registros SESCO en último período sin geometría: {len(sin_geom)}")
    if len(sin_geom):
        print("Cuencas SESCO sin polígono en la capa geo:")
        display(sin_geom.sort_values("agrupador_nombre"))

    cols_resumen = ["cuenca", "tipo", "ubicacion", "produccion", "producto", "periodo_str", "_merge"]
    display(gdf_merged[cols_resumen].sort_values("cuenca"))

resumen_merge(gdf_petroleo, "petroleo")
resumen_merge(gdf_gas, "gas")


## 13. Mapas coropléticos exploratorios (Plotly)

In [ ]:
def mapa_coropletico(gdf_merged: gpd.GeoDataFrame, producto: str, titulo: str):
    plot_df = gdf_merged[gdf_merged.geometry.notna()].copy()
    if plot_df["produccion"].notna().sum() == 0:
        print(f"Sin producción para mapa de {producto}; se omite.")
        return None

    geojson = json.loads(plot_df.to_json())
    # feature id = índice del GeoDataFrame en properties
    plot_df = plot_df.reset_index().rename(columns={"index": "fid"})

    fig = px.choropleth_map(
        plot_df,
        geojson=geojson,
        locations="fid",
        featureidkey="properties.fid",
        color="produccion",
        color_continuous_scale="YlOrRd",
        map_style="open-street-map",
        zoom=3.5,
        center={"lat": -38.5, "lon": -64.0},
        title=titulo,
        hover_data={
            "cuenca": True,
            "tipo": True,
            "ubicacion": True,
            "producto": True,
            "periodo_str": True,
            "produccion": ":,.2f",
            "fid": False,
        },
    )
    fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
    fig.show()
    return fig

fig_pet = mapa_coropletico(
    gdf_petroleo,
    "petroleo",
    f"Petróleo — producción promedio diaria por cuenca ({latest_cuenca['petroleo']})",
)
fig_gas = mapa_coropletico(
    gdf_gas,
    "gas",
    f"Gas — producción promedio diaria por cuenca ({latest_cuenca['gas']})",
)


## 14. Exportar artefactos procesados

In [ ]:
# GeoJSON base (geometría + metadatos de cuenca, sin mezclar productos)
gdf_export = gdf_cuencas[
    ["cuenca", "ubicacion", "tipo", "cuenca_norm", "cuenca_join_key", "geometry"]
].copy()
gdf_export.to_file(GEOJSON_OUT, driver="GeoJSON")
print(f"GeoJSON exportado: {GEOJSON_OUT}")

# Vista previa del join (petróleo y gas en filas separadas)
preview_frames = []
for gdf_m, producto in [(gdf_petroleo, "petroleo"), (gdf_gas, "gas")]:
    prev = pd.DataFrame(gdf_m.drop(columns="geometry", errors="ignore"))
    preview_frames.append(prev)

df_preview = pd.concat(preview_frames, ignore_index=True)
cols_preview = [
    c
    for c in [
        "cuenca",
        "tipo",
        "ubicacion",
        "cuenca_norm",
        "cuenca_join_key",
        "agrupador_nombre",
        "agrupador_nombre_norm",
        "produccion",
        "producto",
        "periodo_str",
        "_merge",
    ]
    if c in df_preview.columns
]
df_preview[cols_preview].to_csv(JOIN_PREVIEW_OUT, index=False)
print(f"Preview join exportado: {JOIN_PREVIEW_OUT}")
print(f"Filas preview: {len(df_preview):,}")


## 15. Conclusiones

| Pregunta | Resultado observado |
|----------|----------------------|
| ¿WKT se leyó correctamente? | Sí: 5 polígonos parseados desde `WKT` sin nulos. |
| ¿CRS definido? | Sí: `EPSG:4326` (el CSV no declara otro CRS). |
| ¿Geometrías válidas? | Revisar salida de `is_valid`; en la muestra actual se esperan polígonos válidos. |
| ¿Coincidencia de nombres con SESCO? | 4/5 por `cuenca_norm` exacto; `AUSTRAL MARINA` (geo) ≠ `AUSTRAL` (SESCO) sin equivalencia manual. |
| ¿Equivalencias necesarias? | `CUENCA_EQUIVALENCIAS_GEO_A_SESCO = {"AUSTRAL MARINA": "AUSTRAL"}` — validar con criterio de negocio. |
| ¿Merge petróleo / gas? | Tras equivalencia, las 5 cuencas geo reciben producción en el último período por producto (`petroleo` → 2025-11, `gas` → 2026-04 según `sesco_latest_periods_by_view.csv`). |
| ¿SESCO sin geometría? | ~12 cuencas en SESCO no están en la capa geo (p. ej. `CAÑADON ASFALTO`, `LOS BOLSONES`, `MALVINAS`, etc.). |
| ¿Listo para Streamlit? | **Parcial:** `cuencas_productivas_geo.geojson` sirve como capa base; el join por producto/período debe hacerse en la app o vía artefactos derivados. |
| ¿Pendiente antes de Streamlit? | Ampliar polígonos a más cuencas SESCO; confirmar equivalencia AUSTRAL; no mezclar petróleo y gas en una sola escala; cablear selector de producto y último período por vista. |

**Artefactos generados:**
- `exploration/data/processed/cuencas_productivas_geo.geojson`
- `exploration/data/processed/cuencas_productivas_sesco_join_preview.csv`

**Trazabilidad:** `cuencas_source`, `SESCO_PATH`, `sesco_latest_periods_by_view.csv`.
